In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join
import pandas as pd

# Import Functions
sys.path.append("../../")

from torchvision import transforms

from src.configs.octdl_config import fn_ori, fn_in, fn_out, data_name
from src.file_manager.filepath import FilePath
from src.display.display_image import show_img_examples_ds, get_image_class_distribution_labels
from src.data_generator.octdl import load_octdl_data_dict

seed=2024
fp = FilePath(data_name=data_name, seed=seed)

fp_ori_data_df = join(fp.get_preprocessed_folder(), fn_ori)
fp_in_data_df = join(fp.get_preprocessed_folder(), fn_in)
fp_out_data_df = join(fp.get_preprocessed_folder(), fn_out)

# Data Exploration

In [ ]:
data_df = pd.read_csv(fp_ori_data_df, index_col=0)
data_df

In [ ]:
data_df[(data_df["disease"]=="AMD") & (data_df["subcategory"]=="early") ]

In [ ]:
data_df["disease"].value_counts()

In [ ]:
data_df["condition"].value_counts()

# New Labels

In [ ]:
# Extract out DME, Drusen and Normal Images
def generate_new_labels(data_df):
    int_mapping = {
        "AMD": 0, "DME": 1, "Drusen": 2, "NO": 3, "ERM": 4, "RVO": 5, "VID": 6, "RAO": 7
    }
    data_df = data_df.copy()
    ori_labels = data_df["disease"]
    drusen_mask = data_df["condition"]=="drusen"
    new_labels = ori_labels.copy()
    new_labels[drusen_mask] = "Drusen"
    data_df["label"] = new_labels
    data_df["int_label"] = data_df["label"].replace(int_mapping)
    data_df["fp"] = data_df["disease"]+"/"+data_df["file_name"]
    return data_df
data_df = generate_new_labels(data_df)
sorted_labels = data_df["label"].unique()
sorted_labels.sort()
print(sorted_labels)
display(data_df["label"].value_counts())
display(data_df["int_label"].value_counts())
# OCTMNIST CLASSSES: ('CNV', 'DME', 'Drusen', 'Normal')

In [ ]:
in_octmnist_df = data_df[data_df["label"].isin(["NO", "Drusen", "DME"])]
out_octmnist_df = data_df[~data_df["label"].isin(["NO", "Drusen", "DME"])]
display(in_octmnist_df["label"].value_counts())
display(out_octmnist_df["label"].value_counts())

In [ ]:
in_octmnist_df.to_csv(fp_in_data_df, index=False)
out_octmnist_df.to_csv(fp_out_data_df, index=False)

# Load Data

In [ ]:
in_data_dict, out_data_dict = load_octdl_data_dict(fp_preprocessed=fp.get_preprocessed_folder())

In [ ]:
show_img_examples_ds(
    in_data_dict["test_df"], num_classes=8, 
    classes=in_data_dict["classes"], ncols=4, transform=False, 
)

In [ ]:
show_img_examples_ds(
    out_data_dict["test_df"], num_classes=8, 
    classes=out_data_dict["classes"], ncols=4, transform=False, 
)

In [ ]:
from torch.utils.data import ConcatDataset
show_img_examples_ds(
    ConcatDataset([in_data_dict["test_df"], out_data_dict["test_df"]]), num_classes=8, 
    classes=in_data_dict["classes"], ncols=4, transform=False, 
)